# N-ATLaS server for the WAZOBIA dashboard

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells in order (Shift+Enter).

You need a Hugging Face token (`hf_...`) from the account that was granted access to `NCAIR1/N-ATLaS`. Read access is enough.

## 1. Install

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes fastapi uvicorn huggingface_hub

## 2. Log in to Hugging Face (paste your token when prompted)

In [ ]:
import getpass
from huggingface_hub import login
login(token=getpass.getpass("HF token (hf_...): "))

## 3. Load N-ATLaS in 4-bit (fits the free T4; takes a few minutes the first time)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "NCAIR1/N-ATLaS"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto"
)
model.eval()
print("Loaded. GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))

## 4. Quick test (checks the model answers before we put a server on it)

In [ ]:
from datetime import datetime

def generate_reply(messages, max_tokens=60):
    text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False,
        date_string=datetime.now().strftime("%d %b %Y"),
    )
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    eos_ids = [i for i in {tokenizer.eos_token_id, eot} if isinstance(i, int) and i >= 0]
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.1,
            repetition_penalty=1.12,
            use_cache=True,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

test = [
    {"role": "system", "content": "You are a translator. Reply with only the Hausa translation of the phrase. No explanation, no quotes, no extra text."},
    {"role": "user", "content": "Translate into Hausa: Good Morning"},
]
print(generate_reply(test))

## 5. Start the API server

This makes a random API key. The dashboard must send it, so strangers who find the URL can't use your GPU.

In [ ]:
import secrets, threading, uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

API_KEY = secrets.token_urlsafe(24)
lock = threading.Lock()   # one generation at a time on the GPU

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class Req(BaseModel):
    messages: list
    max_tokens: int = 60
    model: str | None = None
    stream: bool = False

@app.get("/health")
def health():
    return {"ok": True}

@app.post("/v1/chat/completions")
def chat(req: Req, authorization: str | None = Header(default=None)):
    if authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="bad key")
    with lock:
        reply = generate_reply(req.messages, max_tokens=min(req.max_tokens, 200))
    return {"choices": [{"message": {"role": "assistant", "content": reply}}]}

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
print("Server started on port 8000")
print("API KEY (paste into the dashboard):", API_KEY)

## 6. Make it reachable from your laptop (free Cloudflare tunnel)

Run this cell and wait for the `https://....trycloudflare.com` URL. **The URL changes every time you re-run it**, so paste the new one into the dashboard each session. Keep this notebook tab open; Colab stops idle sessions.

In [ ]:
import subprocess, re, time, os

if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
t0 = time.time()
while time.time() - t0 < 60 and public_url is None:
    line = proc.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        public_url = m.group(0)

print("PUBLIC URL (paste into the dashboard):", public_url)
print("API KEY:", API_KEY)

## 7. Test through the public URL (this is exactly what the dashboard will do)

In [ ]:
import requests, json

r = requests.post(
    public_url + "/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
    json={"messages": test, "max_tokens": 40},
    timeout=60,
)
print(r.status_code)
print(json.dumps(r.json(), ensure_ascii=False, indent=2))